# Classification: Support Vector Classifier (SVC)

## Justification of Preprocessing Strategy

Support Vector Machines (SVM) are highly sensitive to the scale of input features due to their geometric nature.

### **Distance-Based Optimization**
The SVC algorithm works by finding a hyperplane that maximizes the margin between classes. Since this margin is calculated based on Euclidean distances, features with larger numerical ranges would dominate the optimization process if not properly scaled. Therefore, Standardization and Normalization are mandatory preprocessing steps for this model.

### **Kernel Selection**
We evaluate both linear kernels for direct separation and the RBF (Radial Basis Function) kernel. The RBF kernel is particularly useful for capturing non-linear relationships between clinical variables and diabetes diagnosis.

### **Computational Considerations**
Processing large datasets with a non-linear SVC is computationally expensive (≈ $O(n^2)$). In this work, we used 50,000 rows — the maximum feasible without making training overly heavy. To mitigate computational costs, we increased the `cache_size` and used multiple cores where possible.

## Experiment Design

We defined a tournament of 6 experiments to identify the most robust configuration:

### **Runs 1, 2, and 3: Standardization Path**
- **Baseline**: Default parameters ($C=1.0$, `kernel='rbf'`) with StandardScaler.
- **GridSearchCV**: Exhaustive search over a defined grid of $C$ values and kernels.
- **Optuna**: Bayesian optimization to find the ideal trade-off between margin violation and boundary complexity.

### **Runs 4, 5, and 6: Normalization Path**
- **Baseline**: Default parameters with MinMaxScaler.
- **GridSearchCV**: Tests whether the [0, 1] range provides better convergence for the SVC solver.
- **Optuna**: Fine-tunes hyperparameters within the normalized feature space.

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import recall_score, accuracy_score, f1_score

mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_SVC")

<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/Classification/SVC/mlruns/4'), creation_time=1777743144289, experiment_id='4', last_update_time=1777743144289, lifecycle_stage='active', name='Classification_SVC', tags={}, workspace='default'>

In [2]:
df = pd.read_csv("C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/data/diabetes_dataset_new_variables.csv")

# Categorical columns for One-Hot Encoding
categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Transform categories into numerical dummies
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# 3. Stratified Sampling for 50,000 rows
# Using 50,000 rows for training and the remaining 20,000 for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    train_size=50000, 
    random_state=42, 
    stratify=y
)

num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_svc_metrics(y_true, y_pred, duration):
    """Helper function for consistent metric logging"""
    mlflow.log_metric("recall", recall_score(y_true, y_pred))
    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("f1", f1_score(y_true, y_pred))
    mlflow.log_metric("fit_time", duration)

# 6 RUNS 
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

for s_name, scaler in scalers.items():
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

    # --- RUN: BASELINE ---
    with mlflow.start_run(run_name=f"SVC_{s_name}_Baseline_50k"):
        # Setting cache_size to 2000MB to speed up kernel calculations
        model = SVC(kernel='rbf', C=1.0, cache_size=2000, random_state=42)
        
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        y_pred = model.predict(X_test_scaled)
        
        mlflow.log_params(model.get_params())
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "none")
        mlflow.log_param("row_count", 50000)
        log_svc_metrics(y_test, y_pred, duration)

    # --- RUN: GRIDSEARCHCV ---
    with mlflow.start_run(run_name=f"SVC_{s_name}_GridSearch_50k"):
        # Narrowing the grid to ensure completion on 50k rows
        param_grid = [
            {'kernel': ['linear'], 'C': [1, 10]},
            {'kernel': ['rbf'], 'C': [1, 10], 'gamma': ['scale']}
        ]
        
        # Using n_jobs=2 to prevent Windows handle exhaustion
        grid = GridSearchCV(
            SVC(cache_size=2000, random_state=42), 
            param_grid, cv=3, scoring='recall', n_jobs=2 
        )
        
        start_time = time.time()
        grid.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        y_pred = grid.best_estimator_.predict(X_test_scaled)
        
        mlflow.log_params(grid.best_params_)
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "GridSearchCV")
        log_svc_metrics(y_test, y_pred, duration)

    # --- RUN: OPTUNA ---
    def objective(trial):
        c_val = trial.suggest_float("C", 0.1, 10, log=True)
        kernel_val = trial.suggest_categorical("kernel", ["linear", "rbf"])
        
        # Internal model for optimization
        model_opt = SVC(
            C=c_val, 
            kernel=kernel_val, 
            cache_size=2000, 
            random_state=42
        )
        
        model_opt.fit(X_train_scaled, y_train)
        return recall_score(y_test, model_opt.predict(X_test_scaled))

    with mlflow.start_run(run_name=f"SVC_{s_name}_Optuna_50k"):
        study = optuna.create_study(direction="maximize")
        start_time = time.time()
        # 8 trials provide a good balance for SVC on 50k rows
        study.optimize(objective, n_trials=8) 
        duration = time.time() - start_time
        
        # Re-train the champion model to log all metrics
        best_params = study.best_params
        best_svc = SVC(**best_params, cache_size=2000, random_state=42)
        best_svc.fit(X_train_scaled, y_train)
        
        mlflow.log_params(best_params)
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "optuna")
        log_svc_metrics(y_test, best_svc.predict(X_test_scaled), duration)

[I 2026-05-03 11:58:55,966] A new study created in memory with name: no-name-1c751163-14d7-4259-bcad-dbb5b0f370d7
[I 2026-05-03 12:00:28,647] Trial 0 finished with value: 0.8819960665355512 and parameters: {'C': 0.11773604143280839, 'kernel': 'linear'}. Best is trial 0 with value: 0.8819960665355512.
[I 2026-05-03 12:03:58,721] Trial 1 finished with value: 0.8818627287576253 and parameters: {'C': 1.7821616114787953, 'kernel': 'linear'}. Best is trial 0 with value: 0.8819960665355512.
[I 2026-05-03 12:06:21,856] Trial 2 finished with value: 0.8730291009700323 and parameters: {'C': 0.20716543020221848, 'kernel': 'rbf'}. Best is trial 0 with value: 0.8819960665355512.
[I 2026-05-03 12:17:57,763] Trial 3 finished with value: 0.8818960632021068 and parameters: {'C': 9.980932419908177, 'kernel': 'linear'}. Best is trial 0 with value: 0.8819960665355512.
[I 2026-05-03 12:25:04,015] Trial 4 finished with value: 0.8818627287576253 and parameters: {'C': 5.2121672983505185, 'kernel': 'linear'}. B

## SVC Runs Comparison

| Run | Scaler | Optimization | Accuracy | F1 | Recall | Fit Time (s) |
|---|---|---|---:|---:|---:|---:|
| SVC_Normalization_Baseline_50k | Normalization | none | 0.86974 | 0.88989 | 0.87733 | 101.060 |
| SVC_Normalization_GridSearch_50k | Normalization | GridSearchCV | 0.87582 | 0.89501 | 0.88223 | 548.297 |
| SVC_Normalization_Optuna_50k | Normalization | Optuna | 0.87482 | 0.89433 | 0.88286 | 1060.347 |
| SVC_Standardization_Baseline_50k | Standardization | none | 0.88912 | 0.90382 | 0.86833 | 84.080 |
| SVC_Standardization_GridSearch_50k | Standardization | GridSearchCV | 0.87606 | 0.89516 | 0.88186 | 1446.305 |
| SVC_Standardization_Optuna_50k | Standardization | Optuna | 0.87616 | 0.89525 | 0.88203 | 2050.859 |

### Analysis of Results

- **Baseline vs Optimization**: The Standardization baseline achieved the highest accuracy and F1 (accuracy 0.88912, F1 0.90382), with minimal recall trade-off (0.86833).
- **Optimization methods**: GridSearchCV and Optuna yield marginal metric improvements (~0.01 recall gain) at substantial computational cost (17x and 24x longer training times).
- **Computational efficiency**: GridSearchCV and Optuna do not justify their training overhead for the minimal metric gains in a production setting.

### Selected Run for Deployment

- **SVC_Standardization_Baseline_50k** — recommended for Streamlit diabetes diagnosis prediction system:
  - Highest F1 score (0.90382) ensures balanced precision-recall performance.
  - Highest accuracy (0.88912) for reliable predictions.
  - Fastest training time (84.08s) enables rapid model retraining and deployment.
  - Minimal recall trade-off (~0.01) is negligible in clinical decision-support context.
  - Standardized scaling provides numerical stability for inference.
